# ILSA Fine-tuning v2 — Colab A100 Pipeline

**Başlamadan önce (bir kez):**
1. Runtime → Change runtime type → **A100**
2. Sol menüdeki 🔑 **Secrets** bölümüne `GITHUB_TOKEN` adıyla GitHub token'ınızı ekleyin ve "Notebook access"i açın. Token'ı hiçbir hücreye yazmayın.

**Çalıştırma:** Runtime → **Run all**.

**Bağlantı koparsa:** Reconnect → tekrar **Run all**. Her aşamanın çıktısı biter bitmez Drive'a (`MyDrive/ILSA_finetuning_v2_outputs`) kaydedilir; yeniden çalıştırınca Drive'dan geri yüklenir ve tamamlanmış aşamalar atlanır. Korpus çıkarımı belge belge kaydedilir ve kaldığı belgeden devam eder.

**Tek istisna:** LoRA eğitimi (~2-3 saat) yarıda kesilirse eğitim baştan başlar. Eğitim bittiği anda adaptörler Drive'a kaydedilir.

Aşamalar: 0 GPU kontrolü · 1 Kurulum · 2 Eğitim · 3 Checkpoint çıkarımı · 4 Checkpoint seçimi · 5 A0/A1 çıkarımı · 6 Değerlendirme · 7 Tam korpus

## 0. GPU kontrolü

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(gpu)
assert 'A100' in gpu, 'HATA: A100 seçili değil! Runtime → Change runtime type → A100.'

## 1. Kurulum: Drive, repo, Drive'dan geri yükleme

In [ ]:
import os, json, time, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO     = 'dedemerve/ILSA-LLM-Extractor'
BRANCH   = 'feat/extract-v2-resume'
REPO_DIR = Path('/content/ILSA_LLMs')
DRIVE    = Path('/content/drive/MyDrive/ILSA_finetuning_v2_outputs')
RUN_NAME = 'qwen3_8b_r16'
MODEL    = 'Qwen/Qwen3-8B'

# Repo: yoksa klonla (token Colab Secrets'tan okunur, ekrana yazılmaz)
if not (REPO_DIR / 'scripts/finetuning/extract_v2.py').exists():
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = ''
    url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
    r = subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', url, str(REPO_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Repo klonlanamadı (dal: {BRANCH}). Dal GitHub'da var mı, "
                           f"GITHUB_TOKEN secret'ı doğru mu? git çıktısı: {r.stderr.replace(token, '***') if token else r.stderr}")
os.chdir(REPO_DIR)
print('Klasör:', os.getcwd())

V2 = Path('outputs/finetuning/v2')
SUBDIRS = ['runs', 'predictions', 'eval']

def save(*subs):
    """Yerel çıktıları Drive'a kopyalar."""
    for sub in subs or SUBDIRS:
        if (V2 / sub).exists():
            shutil.copytree(V2 / sub, DRIVE / sub, dirs_exist_ok=True)
    print("💾 Drive'a kaydedildi:", ', '.join(subs or SUBDIRS))

def restore():
    """Drive'daki çıktıları yerele geri yükler (tamamlanmış aşamalar atlansın diye)."""
    for sub in SUBDIRS:
        if (DRIVE / sub).exists():
            shutil.copytree(DRIVE / sub, V2 / sub, dirs_exist_ok=True)
    print("♻️  Drive'dan geri yüklendi:", ', '.join(s for s in SUBDIRS if (DRIVE / s).exists()) or 'hiçbir şey (ilk çalıştırma)')

def sh(cmd):
    """Kabuk komutunu çalıştırır; hata olursa Run all'u durdurur."""
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get('_exit_code', 0)
    if code:
        raise RuntimeError(f'Komut başarısız oldu (çıkış kodu {code}). Yukarıdaki hatayı okuyun.')

def stage(name, out, cmd, *save_subs):
    """Çıktı zaten varsa atlar; yoksa çalıştırır ve hemen Drive'a kaydeder."""
    out = Path(out)
    if out.exists():
        print(f'✓ {name}: zaten tamamlanmış, atlandı ({out})')
        return
    print(f'▶ {name} başlıyor...')
    sh(cmd)
    assert out.exists(), f'{name}: beklenen çıktı oluşmadı: {out}'
    save(*save_subs)
    print(f'✓ {name} tamamlandı')

DRIVE.mkdir(parents=True, exist_ok=True)
restore()

In [ ]:
# Paketler: eğitim ortamı + vLLM (çıkarım). Ağır işler ayrı süreçlerde çalıştığı için
# ikisi aynı oturumda kurulabilir; pip'in "dependency conflict" uyarıları zararsızdır.
sh('pip install -q -r requirements-finetuning-v2.txt')
sh('pip install -q vllm==0.31.0 transformers==5.17.0 openai==3.26.1')
print('Paketler hazır.')

## 2. LoRA eğitimi (~2-3 saat)

`run_manifest.json` Drive'dan geri yüklendiyse eğitim atlanır. Eğitim yeniden yapılırsa, eski adaptörlere ait tahminler Drive'da `_eski/` klasörüne taşınır ki yeni adaptörlerle karışmasın.

In [ ]:
MANIFEST = V2 / f'runs/{RUN_NAME}/run_manifest.json'
STALE = [f'predictions/{RUN_NAME}_checkpoint-*', 'predictions/A1_*',
         'eval/checkpoint_selection_val.json', 'eval/A1_*']

if MANIFEST.exists():
    print('✓ Eğitim zaten tamamlanmış, atlandı.')
else:
    # Eski adaptörlerin sonuçlarını kenara al
    archive = DRIVE / '_eski' / time.strftime('%Y%m%d_%H%M')
    for pat in STALE:
        for f in V2.glob(pat):
            f.unlink()
        for f in DRIVE.glob(pat):
            dest = archive / f.relative_to(DRIVE)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(f), str(dest))

    # Bellek testi: peak_gb < 38 olmalı
    sh(f'python3 scripts/finetuning/train_lora_v2.py --model {MODEL} '
       f'--out outputs/finetuning/v2/runs/{RUN_NAME} --epochs 3 --r 16 --batch-size 1 --memory-probe')

    stage('LoRA eğitimi', MANIFEST,
          f'python3 scripts/finetuning/train_lora_v2.py --model {MODEL} '
          f'--out outputs/finetuning/v2/runs/{RUN_NAME} --epochs 3 --r 16 --batch-size 1 --grad-accum 4',
          'runs')

m = json.loads(MANIFEST.read_text())
assert m['adapter_reload_verified'], 'Adapter reload doğrulaması BAŞARISIZ!'
CHECKPOINTS = [Path(p).name for p in m['epoch_checkpoints']]
missing = [c for c in CHECKPOINTS if not (V2 / f'runs/{RUN_NAME}/{c}').exists()]
assert not missing, f'Adaptör klasörleri eksik: {missing}'
print('Checkpoint\'lar:', CHECKPOINTS)

## 3. Her checkpoint için val çıkarımı (checkpoint seçimi için)

In [ ]:
for ckpt in CHECKPOINTS:
    arm = f'{RUN_NAME}_{ckpt}'
    stage(f'Val çıkarımı {ckpt}', V2 / f'predictions/{arm}_val.json',
          f'python3 scripts/finetuning/extract_v2.py --mode chunked --backend vllm --model {MODEL} '
          f'--adapter outputs/finetuning/v2/runs/{RUN_NAME}/{ckpt} --split val --arm {arm}',
          'predictions')

## 4. En iyi checkpoint seçimi

In [ ]:
SEL = V2 / 'eval/checkpoint_selection_val.json'
stage('Checkpoint seçimi', SEL, 'python3 scripts/finetuning/select_checkpoint.py --split val', 'eval')
sel = json.loads(SEL.read_text())
BEST_ADAPTER = sel['selected']['adapter']
print('SEÇİLEN ADAPTER:', BEST_ADAPTER)
print('Strict F1:', sel['selected']['strict_f1'])

## 5. A0 (base) ve A1 (LoRA) val çıkarımı

In [ ]:
stage('A0 çıkarımı', V2 / 'predictions/A0_base_qwen3_chunked_val.json',
      f'python3 scripts/finetuning/extract_v2.py --mode chunked --backend vllm --model {MODEL} '
      f'--split val --arm A0_base_qwen3_chunked',
      'predictions')

stage('A1 çıkarımı', V2 / 'predictions/A1_qwen3_lora_selected_val.json',
      f'python3 scripts/finetuning/extract_v2.py --mode chunked --backend vllm --model {MODEL} '
      f'--adapter {BEST_ADAPTER} --split val --arm A1_qwen3_lora_selected',
      'predictions')

## 6. A0 ve A1 değerlendirmesi

In [ ]:
stage('A0 değerlendirme', V2 / 'eval/A0_base_qwen3_val.json',
      'python3 scripts/finetuning/eval_extraction.py '
      '--pred outputs/finetuning/v2/predictions/A0_base_qwen3_chunked_val.json '
      '--split val --label A0 --out outputs/finetuning/v2/eval/A0_base_qwen3_val.json',
      'eval')
stage('A1 değerlendirme', V2 / 'eval/A1_qwen3_lora_val.json',
      'python3 scripts/finetuning/eval_extraction.py '
      '--pred outputs/finetuning/v2/predictions/A1_qwen3_lora_selected_val.json '
      '--split val --label A1 --out outputs/finetuning/v2/eval/A1_qwen3_lora_val.json',
      'eval')

a0_s = json.loads((V2 / 'eval/A0_base_qwen3_val.json').read_text())['A0']['summary']
a1_s = json.loads((V2 / 'eval/A1_qwen3_lora_val.json').read_text())['A1']['summary']
print('=== SONUÇ ÖZETİ ===')
print(f"A0  strict={a0_s['strict']['f1']}  lenient={a0_s['lenient']['f1']}  legacy={a0_s['legacy_greedy_construct']['f1']}")
print(f"A1  strict={a1_s['strict']['f1']}  lenient={a1_s['lenient']['f1']}  legacy={a1_s['legacy_greedy_construct']['f1']}")
print(f'Seçilen adapter: {BEST_ADAPTER}')
if a1_s['strict']['f1'] > a0_s['strict']['f1']:
    print('H1 DESTEKLENDİ: A1 strict F1 > A0 strict F1')
else:
    print('H1 desteklenmedi (A1 <= A0 strict F1) — raporlanabilir, başarısızlık değil')

## 7. Tam korpus çıkarımı — A1_fullcorpus_qwen3_lora (~7-9 saat)

Her belge biter bitmez Drive'daki `predictions/A1_fullcorpus_qwen3_lora_corpus.ckpt.jsonl` dosyasına yazılır. Bağlantı koparsa **Run all** ile kaldığı belgeden devam eder.

In [ ]:
import pandas as pd
inv = pd.read_csv('outputs/finetuning/source_recovery/full_source_inventory.csv', dtype=str)
n_corpus = (inv['text_extracted'].str.strip().str.lower() == 'true').sum()
print(f'Metin çıkarılmış çalışma sayısı: {n_corpus}')
print(f'Kullanılacak adapter: {BEST_ADAPTER}')

CORPUS_OUT  = V2 / 'predictions/A1_fullcorpus_qwen3_lora_corpus.json'
CORPUS_CKPT = DRIVE / 'predictions/A1_fullcorpus_qwen3_lora_corpus.ckpt.jsonl'
if CORPUS_CKPT.exists():
    print(f'Daha önce tamamlanan belge sayısı: {sum(1 for l in CORPUS_CKPT.open() if l.strip())}')

stage('Tam korpus çıkarımı', CORPUS_OUT,
      f'python3 scripts/finetuning/extract_v2.py --mode chunked --backend vllm --model {MODEL} '
      f'--adapter {BEST_ADAPTER} --split corpus --arm A1_fullcorpus_qwen3_lora '
      f'--checkpoint "{CORPUS_CKPT}"',
      'predictions')

preds = json.loads(CORPUS_OUT.read_text())['predictions']
print(f"Çalışma sayısı : {len(preds)}")
print(f"ok             : {sum(p['status'] == 'ok' for p in preds)}")
print(f"missing_source : {sum(p['status'] == 'missing_source' for p in preds)}")
print(f"Toplam çift    : {sum(len(p['pairs']) for p in preds)}")
print(f"Drive kayıt    : {DRIVE / 'predictions'}")